# *atommovr* tutorial: figure generation

This notebook contains code to a) reproduce the figures presented in the paper, and b) load the data displayed in the paper.

Please note that many of the figures take a long time to run (>1 hr). 
- A runtime estimate for each figure is shown in the corresponding box. 
- Unless otherwise noted, runtimes are for a MacBook Pro (Apple M1 chip) with 8 GB RAM running on Sonoma 14.2.1.

Additionally, please note that the curve fitting and plotting code is intended for the existing experiments. 
- Modifications to experimental parameters may require corresponding modifications to the fitting code and/or plot limits, scales, colorbar ranges, etc.
- You can optionally turn off curve fitting with the `do_curve_fits` parameter.

In [ ]:
import ast
import pickle
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.ticker import NullFormatter, NullLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.optimize import curve_fit
from scipy.ndimage import gaussian_filter

import atommovr.utils as movr
from atommovr.utils.benchmarking import Benchmarking
from atommovr.utils.customize import _quantumgray, _quantumviolet, _nikhilgreen, _nikhilblue, _nikhilorange
import atommovr.algorithms as algos
from atommovr.algorithms.source.scaling_lower_bound import calculate_Zstar_better

do_curve_fits: bool = True

## Figure 2

#### Loading data from the paper

In [ ]:
# Fig 2a

saved_bench_2a = Benchmarking()
saved_bench_2a.load('fig2a_final_260505.nc')
results_fig2a = saved_bench_2a.benchmarking_results

start_ind = 1
s_10 = results_fig2a['success rate'].to_numpy()[0,0,start_ind:,0,0,0]
s_100 = results_fig2a['success rate'].to_numpy()[0,0,start_ind:,1,0,0]
s_1000 = results_fig2a['success rate'].to_numpy()[0,0,start_ind:,2,0,0]
s_10000 = results_fig2a['success rate'].to_numpy()[0,0,start_ind:,3,0,0]
raw_n_targets = results_fig2a['n targets'].to_numpy()[0,0,:,0,0,0]
n_targets = np.array([ast.literal_eval(x) for x in raw_n_targets], dtype=np.int64)

n_shots_2a = 500 
lifetimes = [10, 100, 1000, 10000]
target_sizes = []
for i in range(start_ind, len(n_targets)):
    target_size = n_targets[i][0]
    target_sizes.append(target_size)

# for Fig 2b

saved_bench_2b = Benchmarking()
saved_bench_2b.load('fig2b_final_260505.nc')
results_fig2b = saved_bench_2b.benchmarking_results

r1 = results_fig2b['filling fraction'].to_numpy()[0,0,0,0,0,0]
r2 = results_fig2b['filling fraction'].to_numpy()[0,0,0,0,0,1]
r3 = results_fig2b['filling fraction'].to_numpy()[0,0,0,0,0,2]

def parse_maybe_stringified_list(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

r3 = np.asarray(parse_maybe_stringified_list(r3), dtype=np.float64)
r2 = np.asarray(parse_maybe_stringified_list(r2), dtype=np.float64)
r1 = np.asarray(parse_maybe_stringified_list(r1), dtype=np.float64)
s = results_fig2b['success rate'].to_numpy()[0,0,0,0,0,:]

#### Rerunning Fig. 2a

Runtime: ~ 9 hrs for `n_shots_2a = 500`.

In [ ]:
n_shots_2a = 500

# experimental parameters
lifetimes = [10, 100, 1000, 10000]
new_err_models = []
for lifetime in lifetimes:
    new_model = movr.UniformVacuumTweezerError(putdown_time = 200e-6,
                                               pickup_time= 200e-6,
                                               accel_time = 12.5e-6,
                                               decel_time = 12.5e-6,
                                               pickup_fail_rate=0,
                                               putdown_fail_rate=0,
                                               lifetime=lifetime)
    new_err_models.append(new_model)

target_configs = [movr.Configurations.MIDDLE_FILL]
error_models_list = new_err_models
algorithms = [algos.BCv2()]
phys_params_list = [movr.PhysicalParams(AOD_speed = 0.2, spacing = 5e-6)]
system_size_range = [8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88]

bench_2a = Benchmarking(algos = algorithms,
                        target_configs = target_configs,
                        error_models_list = error_models_list,
                        phys_params_list = phys_params_list,
                        sys_sizes = system_size_range,
                        n_shots = n_shots_2a)
# running sweep
bench_2a.run()

In [ ]:
bench_2a.benchmarking_results

In [ ]:
# extracting success rate results for each atom lifetime value
start_ind = 1
s_10 = bench_2a.benchmarking_results['success rate'].to_numpy()[0,0,start_ind:,0,0,0]
s_100 = bench_2a.benchmarking_results['success rate'].to_numpy()[0,0,start_ind:,1,0,0]
s_1000 = bench_2a.benchmarking_results['success rate'].to_numpy()[0,0,start_ind:,2,0,0]
s_10000 = bench_2a.benchmarking_results['success rate'].to_numpy()[0,0,start_ind:,3,0,0]
n_atoms = bench_2a.benchmarking_results['n targets'].to_numpy()[0,0,:,0,0,0]
target_sizes = []
for i in range(start_ind, len(n_atoms)):
    target_size = n_atoms[i][0]
    target_sizes.append(target_size)

#### Rerunning Fig. 2b

Runtime: ~ 2 hrs for `n_shots_2b = 1000`.

In [ ]:
n_shots_2b = 1000

# experimental parameters
vacuum_limited_model = movr.UniformVacuumTweezerError(lifetime = 15,
                                                      putdown_time = 200e-6,
                                                      pickup_time = 200e-6,
                                                      accel_time = 12.5e-6,
                                                      decel_time = 12.5e-6,
                                                      pickup_fail_rate = 0,
                                                      putdown_fail_rate = 0,
                                                      accel_fail_rate = 0,
                                                      decel_fail_rate = 0)

target_configs = [movr.Configurations.MIDDLE_FILL]
rounds_list = [1,2,3]
error_models_list = [vacuum_limited_model]
algorithms = [algos.Hungarian()]
system_size_range = [50]
phys_params_list = [movr.PhysicalParams(AOD_speed = 0.2, spacing = 5e-6, middle_size = [36,36])]

# running sweep
bench_2b = Benchmarking(
    algos = algorithms,
    target_configs = target_configs,
    error_models_list = error_models_list,
    phys_params_list = phys_params_list,
    sys_sizes = system_size_range,
    rounds_list = rounds_list,
    n_shots = n_shots_2b
)
bench_2b.run()

In [ ]:
bench_2b.benchmarking_results

In [ ]:
# extracting filling fraction results for different number of rearrangement rounds
r1 = bench_2b.benchmarking_results['filling fraction'].to_numpy()[0,0,0,0,0,0]
r2 = bench_2b.benchmarking_results['filling fraction'].to_numpy()[0,0,0,0,0,1]
r3 = bench_2b.benchmarking_results['filling fraction'].to_numpy()[0,0,0,0,0,2]
s = bench_2b.benchmarking_results['success rate'].to_numpy()[0,0,0,0,0,:]

#### Plotting data for Fig. 2a and Fig. 2b (must run 'Loading data from the paper' box or 'Rerunning Fig. 2x' first).

In [ ]:
fig2, ax = plt.subplots(1,2, figsize=(10,4.25))

# useful functions
def sigmoid(x,a,b):
    return 1/(1+np.exp(b*(x-a)))
def logfunc(x,a,b,c):
    return c*np.log(a*(x-b))
def pwlaw(x,a,b,c):
    return c*((x-a)**b)

## fig 2a ##
xdata = target_sizes
alpha_marker = 0.4
alpha_border = 0.7
colors = [_nikhilgreen, _quantumgray, _nikhilblue, _quantumviolet]
facecolors = colors

shapemarkers = ["v", "o", "s", "d"]
nummarkers = ['$10^{1}$', '$10^{2}$','$10^{3}$','$10^{4}$' ]
markers = shapemarkers
mark_size = 6/0.8

edgecolors = colors
x_axis = np.linspace(1,6000, 100)

labels_full = [r'$t_{\text{lifetime}} = 10~\text{s}$', 
          r'$t_{\text{lifetime}} = 100~\text{s}$', 
          r'$t_{\text{lifetime}} = 1000~\text{s}$', 
          r'$t_{\text{lifetime}} = 10000~\text{s}$']
labels_num = ['10 s', '100 s', '1000 s', '10000 s']
labels_short = [r't=$10^{1}$ s', 
                r't=$10^{2}$ s', 
                r't=$10^{3}$ s', 
                r't=$10^{4}$ s']
labels = labels_short

# curve fits
if do_curve_fits:
    popt10, pcov10 = curve_fit(sigmoid, xdata, s_10, p0 =[300,0.03], maxfev = 1000000)
    ax[0].plot(x_axis, sigmoid(x_axis, *popt10), c=colors[0])
    perr10 = np.sqrt(np.diag(pcov10))

    popt100, pcov100 = curve_fit(sigmoid, xdata, s_100, p0 =[1000,0.003])
    ax[0].plot(x_axis, sigmoid(x_axis, *popt100),c=colors[1])
    perr100 = np.sqrt(np.diag(pcov100))

    popt1000, pcov1000 = curve_fit(sigmoid, xdata, s_1000, p0 =[2000,0.001])
    ax[0].plot(x_axis, sigmoid(x_axis, *popt1000), c=colors[2])
    perr1000 = np.sqrt(np.diag(pcov1000))

    popt10000, pcov10000 = curve_fit(sigmoid, xdata, s_10000, p0 =[3900,0.001])
    ax[0].plot(x_axis, sigmoid(x_axis, *popt10000),c=colors[3])
    perr10000 = np.sqrt(np.diag(pcov10000))


ymin, ymax = ax[0].get_ylim() 
#vertical lines
if do_curve_fits:
    ax[0].vlines([popt10[0], popt100[0],popt1000[0]], colors=colors[:3], ymin = -5, ymax = 5, alpha = alpha_marker, linestyles='dashed')
    ax[0].vlines(popt10000[0], colors=colors[3], ymin = -5, ymax = 0.55, alpha = alpha_marker, linestyles='dashed')
ax[0].set_ylim(ymin, ymax)
ax[0].errorbar(target_sizes, s_10,    yerr=1/np.sqrt(n_shots_2a)*s_10,    c=colors[0],label = labels[0], fmt = markers[0], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[0], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[0], alpha=alpha_border))
ax[0].errorbar(target_sizes, s_100,   yerr=1/np.sqrt(n_shots_2a)*s_100,   c=colors[1],label = labels[1], fmt = markers[1], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[1], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[1], alpha=alpha_border))
ax[0].errorbar(target_sizes, s_1000,  yerr=1/np.sqrt(n_shots_2a)*s_1000,  c=colors[2],label = labels[2], fmt = markers[2], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[2], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[2], alpha=alpha_border))
ax[0].errorbar(target_sizes, s_10000, yerr=1/np.sqrt(n_shots_2a)*s_10000, c=colors[3],label = labels[3], fmt = markers[3], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[3], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[3], alpha=alpha_border))

if do_curve_fits:
    inset_ax = inset_axes(ax[0], width="12%", height="24%", loc="upper right")  # Width and height as a percentage of main plot

    lifetimes = np.array(lifetimes)
    half_probs = [popt10[0], popt100[0],popt1000[0],popt10000[0]]

    x_vals = np.linspace(0.005,10, 100)

    popt, pcov = curve_fit(logfunc, lifetimes, half_probs, p0 =[0.01, 0, 1500], maxfev = 1000)

    inset_ax.errorbar(lifetimes/1e3, 
                half_probs, 
                yerr = [perr10[0],perr100[0],perr1000[0],perr10000[0]],
                fmt = 'o', color=_nikhilblue)

    inset_ax.plot(x_vals, logfunc(x_vals*1e3, *popt), color=_nikhilorange)
    inset_ax.set_ylabel('N [$10^{3}$]')
    inset_ax.set_xlabel(r't [$10^{3}$ s]',labelpad = 1)
    inset_ax.set_yticks([1000,5000,10000])
    inset_ax.set_yticklabels([1,5,10])
    inset_ax.set_xticks([0,5,10])
    inset_ax.set_xticklabels([0,5,10])
    inset_ax.grid(True, which="major", linewidth=0.8)

ax[0].legend(loc = 'lower right')
ax[0].set_ylabel("Success rate")
ax[0].set_xlabel("Number of atoms")
ax[0].grid(True, which="major", linewidth=0.8)
ax[0].set_axisbelow(True)

## Fig 2b ##

bins = np.linspace(85, 100, 25)
labels = ['3 rounds','2 rounds','1 round']
linestyles = ['-','--']
edgecolors = ["white", "gray", "black"]
fillcolors = [_nikhilblue, _nikhilorange, _nikhilgreen]
ab_size = 15 # size of a and b labels in combined figure

counts, bins, patches = ax[1].hist(100*np.array(r3), bins=bins,label=f'3 rounds', alpha = 0.6, color = fillcolors[0], bottom = 0.5)
counts, bins, patches = ax[1].hist(100*np.array(r2), bins=bins,label=f'2 rounds', alpha = 0.6, color = fillcolors[2], bottom = 0.5)
counts, bins, patches = ax[1].hist(100*np.array(r1), bins=bins,label=f'1 round', alpha = 0.6, color = fillcolors[1], bottom = 0.5)
ax[1].set_xlabel("Filling fraction [%]")
ax[1].set_ylabel("Counts")
ax[1].set_yscale('log')
ax[1].set_xticks([85,90,95,100])
ax[1].set_xticklabels([85,90,95,100])
ax[1].legend()
ax[1].grid(True, which="major", linewidth=0.8)
ax[1].set_axisbelow(True)
ax[1].minorticks_off()

plt.gcf().text(0.02, 0.9, r'$\text{a}$', fontsize=ab_size)
plt.gcf().text(0.52, 0.9, r'$\text{b}$', fontsize=ab_size)
plt.subplots_adjust(bottom=0.15, right = 0.97, left = 0.1)
plt.show()

## Figure 3

#### Loading data from the paper

In [ ]:
saved_bench_3 = Benchmarking()
saved_bench_3.load('fig3_final_260505.nc')

hun = saved_bench_3.benchmarking_results['time'].to_numpy()[0,0,:,0,0,0]*1e3 # converting from s to ms
parhun = saved_bench_3.benchmarking_results['time'].to_numpy()[1,0,:,0,0,0]*1e3
bc = saved_bench_3.benchmarking_results['time'].to_numpy()[2,0,:,0,0,0]*1e3

raw_n_targets = saved_bench_3.benchmarking_results['n targets'].to_numpy()[0,0,:,0,0,0]
targets = np.array([ast.literal_eval(x) for x in raw_n_targets], dtype=np.int64)
n_atoms = []
for tnum in targets:
    n_atoms.append(tnum[0])
n_atoms = np.array(n_atoms)

euclidean_array = np.load('data/fig3_zstar_final_260508.npy')
grid_array = np.load('data/fig3_zstargrid_final_260508.npy')

n_shots_3 = 100

avg_vel = 0.1 # m/s
spacing = 5e-6 # 5 um
accel_time = 0
pickup_time = 0

params = movr.PhysicalParams(AOD_speed = avg_vel, spacing = spacing, loading_prob = 0.5)

l6_eucl = np.mean(euclidean_array[0,:,:], axis=1)*(params.spacing/(params.AOD_speed) + 2*accel_time + 2*pickup_time)*1e3 # ms
l6_grid = np.mean(grid_array[0,:,:], axis=1)*(params.spacing/(params.AOD_speed) + 2*accel_time + 2*pickup_time)*1e3 # ms
l6_eucl_std = np.std(euclidean_array[0,:,:], axis=1)*(params.spacing/(params.AOD_speed) + 2*accel_time + 2*pickup_time)*1e3 # ms
l6_grid_std = np.std(grid_array[0,:,:], axis=1)*(params.spacing/(params.AOD_speed)+ 2*accel_time + 2*pickup_time)*1e3 # ms

#### Rerunning Fig. 3

Runtime: ~1.5 hrs for `n_shots_3 = 100`.

In [ ]:
n_shots_3 = 100

# experimental parameters
bench_3 = Benchmarking()

with open('data/scaling_250203.pkl', 'rb') as file: 
    results_fig3 = pickle.load(file)

bench_3.load_params_from_dataset(results_fig3)
bench_3.algos = [algos.Hungarian(), algos.ParallelHungarian(), algos.BCv2()]

avg_vel = 0.1 # m/s
spacing = 5e-6 # 5 um
accel_time = 0
pickup_time = 0

bench_3.phys_params_list = [movr.PhysicalParams(AOD_speed = avg_vel, spacing = spacing, loading_prob = 0.5)]

bench_3.error_models_list = [movr.ZeroNoise(putdown_time = pickup_time, 
                                            pickup_time = pickup_time, 
                                            accel_time = accel_time, 
                                            decel_time = accel_time)]

# running sweep
bench_3.n_shots = n_shots_3
bench_3.run()


## lower bound extraction ##

load_probs = [0.5]
params = bench_3.phys_params_list[0] # NB: this code assumes there is only a single error model
error_model = bench_3.error_models_list[0] # NB: this code assumes there is only a single error model
n_shots = bench_3.n_shots
sizes = bench_3.system_size_range
n_species = bench_3.tweezer_array.n_species

euclidean_array = np.zeros([len(load_probs),len(sizes), n_shots])
grid_array = np.zeros([len(load_probs),len(sizes), n_shots])
max_size = max(sizes)
for si, size in enumerate(sizes):
    for li, load_prob in enumerate(load_probs):
        middle_size = [size, size]
        n = int(np.ceil(np.sqrt((size**2)/load_prob)))
        array = movr.AtomArray(shape=[n,n], params=params, error_model=error_model, n_species = n_species)
        if n_species == 1:
            array.generate_target(movr.Configurations.MIDDLE_FILL, middle_size = middle_size)
        elif n_species == 2:
            array.generate_target(movr.Configurations.ZEBRA_HORIZONTAL, middle_size = middle_size)
        array.params.loading_prob = load_prob
        for i in range(n_shots):
            n_atoms_sufficient = False
            count = 0
            while not n_atoms_sufficient and count < 1000:
                array.load_tweezers()
                if n_species == 2:
                    if np.sum(array.matrix[:,:,0]) >= np.sum(array.target[:,:,0]) and np.sum(array.matrix[:,:,1]) >= np.sum(array.target[:,:,1]):
                        n_atoms_sufficient = True
                elif n_species == 1:
                    if np.sum(array.matrix[:,:,0]) >= np.sum(array.target[:,:,0]):
                        n_atoms_sufficient = True
                count += 1

            new_dist = calculate_Zstar_better(array.matrix, array.target, metric = 'euclidean', n_species = n_species)
            euclidean_array[li,si,i] = new_dist
            grid_dist = calculate_Zstar_better(array.matrix, array.target, metric = 'moves', n_species = n_species)
            grid_array[li,si,i] = grid_dist

In [ ]:
# extracting rearrangement times for each algorithm
hun = bench_3.benchmarking_results['time'].to_numpy()[0,0,:,0,0,0]*1e3 # converting from s to ms
parhun = bench_3.benchmarking_results['time'].to_numpy()[1,0,:,0,0,0]*1e3 # ms
bc = bench_3.benchmarking_results['time'].to_numpy()[2,0,:,0,0,0]*1e3 # ms

l6_eucl = np.mean(euclidean_array[0,:,:], axis=1)*(params.spacing/(params.AOD_speed) + 2*accel_time + 2*pickup_time)*1e3 # ms
l6_grid = np.mean(grid_array[0,:,:], axis=1)*(params.spacing/(params.AOD_speed) + 2*accel_time + 2*pickup_time)*1e3 # ms
l6_eucl_std = np.std(euclidean_array[0,:,:], axis=1)*(params.spacing/(params.AOD_speed) + 2*accel_time + 2*pickup_time)*1e3 # ms
l6_grid_std = np.std(grid_array[0,:,:], axis=1)*(params.spacing/(params.AOD_speed)+ 2*accel_time + 2*pickup_time)*1e3 # ms

targets = bench_3.benchmarking_results['n targets'].to_numpy()[0,0,:,0,0,0]
n_atoms = []
for tnum in targets:
    n_atoms.append(tnum[0])
n_atoms = np.array(n_atoms)

#### Plotting data for Fig. 3 (must run one of the 'Loading data from the paper' or 'Rerunning Fig. 3' boxes first).

In [ ]:
fig3, ax = plt.subplots(1,1, figsize=(5.75,4.95))

# plot params
alpha_marker = 0.4
alpha_border = 0.7
colors = [_nikhilgreen, _quantumgray, _nikhilblue, _quantumviolet, _nikhilorange, [0.4, 0.2, 0.7]]
facecolors = colors

shapemarkers = ["v", "s", "o", "d", '^', "P"]
markers = shapemarkers
mark_size = 8

edgecolors = colors
x_axis = np.linspace(np.min(n_atoms), np.max(n_atoms), 100)

labels_full = ['Hungarian', 
               'BalCompact', 
               r'Z$*$', 
               r'Z$*$ (grid)', 
               'ParHungarian', 
               'ParLBAP']
labels = labels_full

def pwlaw(x,b,c):
    return c*((x)**b)

# removing nans in case some exist
mask_parhun = ~np.isnan(parhun)
mask_hun = ~np.isnan(hun)
mask_bc = ~np.isnan(bc)
if do_curve_fits:
    poptparh, pcovparh = curve_fit(pwlaw, n_atoms[mask_parhun], parhun[mask_parhun], sigma = 1/np.sqrt(n_shots_3)*parhun[mask_parhun], p0 =[1.1, 0.01], maxfev = 100000)
    perrparh = np.sqrt(np.diag(pcovparh))
    poptbc, pcovbc = curve_fit(pwlaw, n_atoms[mask_bc], bc[mask_bc], sigma = 1/np.sqrt(n_shots_3)*bc[mask_bc], p0 =[1,0.01], maxfev = 100000)
    perrbc = np.sqrt(np.diag(pcovbc))
    popthun, pcovhun = curve_fit(pwlaw, n_atoms[mask_hun], hun[mask_hun], sigma = 1/np.sqrt(n_shots_3)*hun[mask_hun], p0 =[1.6,0.1], maxfev = 100000)
    perrhun = np.sqrt(np.diag(pcovhun))

    ax.plot(x_axis, pwlaw(x_axis, *popthun),c=colors[0])
    ax.plot(x_axis, pwlaw(x_axis, *poptbc),c=colors[1])
    ax.plot(x_axis, pwlaw(x_axis, *poptparh),c=colors[4])

ax.errorbar(n_atoms, hun,     yerr=1/np.sqrt(n_shots_3)*hun,     c=colors[0],label = labels[0], fmt = markers[0], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[0], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[0], alpha=alpha_border))
ax.errorbar(n_atoms, bc,      yerr=1/np.sqrt(n_shots_3)*bc,      c=colors[1],label = labels[1], fmt = markers[1], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[1], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[1], alpha=alpha_border))
ax.errorbar(n_atoms, parhun,  yerr=1/np.sqrt(n_shots_3)*parhun,  c=colors[4],label = labels[4], fmt = markers[4], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[4], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[4], alpha=alpha_border))


if do_curve_fits:
    popt_euc, pcov_euc = curve_fit(pwlaw, n_atoms, l6_eucl, sigma = l6_eucl_std,absolute_sigma=True,p0 =[1,0.01], maxfev = 100000)
    perr_euc = np.sqrt(np.diag(pcov_euc))
    popt_gr, pcov_gr = curve_fit(pwlaw, n_atoms, l6_grid, sigma=l6_grid_std, absolute_sigma=True, p0 =[1.6,0.1], maxfev = 100000)
    perr_gr = np.sqrt(np.diag(pcov_gr))

    ax.plot(x_axis, pwlaw(x_axis, *popt_gr),c=colors[3])
    ax.plot(x_axis, pwlaw(x_axis, *popt_euc),c=colors[2])

    print(f"Z:{popt_euc[0]:.3f} +/- {perr_euc[0]:.3f}")
    print(f"Z grid:{popt_gr[0]:.3f} +/- {perr_gr[0]:.3f}")

ax.errorbar(n_atoms, l6_grid, yerr=1/np.sqrt(n_shots_3)*l6_grid, c=colors[3],label = labels[3], fmt = markers[3], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[3], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[3], alpha=alpha_border))
ax.errorbar(n_atoms, l6_eucl, yerr=1/np.sqrt(n_shots_3)*l6_eucl, c=colors[2],label = labels[2], fmt = markers[2], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[2], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[2], alpha=alpha_border))

text_size = 10
ax.legend(prop={'size': text_size})
ax.set_yscale('log')
ax.set_xscale('log')
ax.set_xticks([100, 200, 400])
ax.set_xticklabels([r'$10^{2}$', r'$2\times 10^{2}$', r'$4\times 10^{2}$'],fontsize = text_size)
ax.set_ylabel('Rearrangement time [ms]')
ax.set_xlabel('Number of atoms')
ax.tick_params(axis='both', direction='in', length=6, width=0.5)
ax.grid(True, which="major", linewidth=0.8)


plt.subplots_adjust(bottom=0.15, right = 0.85, left = 0.15, top = 0.95)

if do_curve_fits:
    print(f"BC:{poptbc[0]:.3f} +/- {perrbc[0]:.3f}")
    print(f"Hun:{popthun[0]:.3f} +/- {perrhun[0]:.3f}")
    print(f"ParHun:{poptparh[0]:.3f} +/- {perrparh[0]:.3f}")

## Figure 4

#### Loading data from the paper

In [ ]:
saved_bench_4 = Benchmarking()
saved_bench_4.load('fig4_final_260505')
results_fig4 = saved_bench_4.benchmarking_results

n_shots_4 = 400
tweezer_errors = np.logspace(-3.301, -2.301, 20)[::-1]
vac_lifetimes = np.logspace(0.301, 2.301, 20)
X, Y = np.meshgrid(np.array(tweezer_errors)*100, np.array(vac_lifetimes))

rates = results_fig4['success rate'].to_numpy()[1,0,0,:,0,0]
rates_arr = results_fig4['success rate'].to_numpy()[:,0,0,:,0,0]

#### Rerunning Fig. 4

Runtime: ~7 hrs for `n_shots_4 = 400`.

In [ ]:
n_shots_4 = 2

# experimental parameters
tweezer_errors = np.logspace(-3.301, -2.301, 20)[::-1]
vac_lifetimes = np.logspace(0.301, 2.301, 20)
err_models = []
for tw_err in tweezer_errors:
    for vac_l in vac_lifetimes:
        errmol = movr.UniformVacuumTweezerError(putdown_fail_rate=tw_err/2, 
                                                pickup_fail_rate=tw_err/2, 
                                                lifetime=vac_l,
                                                pickup_time = 200e-6,
                                                putdown_time = 200e-6,
                                                accel_time = 12.5e-6,
                                                decel_time = 12.5e-6)
        err_models.append(errmol)

phys_params_list = [movr.PhysicalParams(AOD_speed = 0.2, spacing = 5e-6)]
algorithms = [algos.Hungarian(),algos.BCv2()]
sizes = [16]

target_configs = [movr.Configurations.MIDDLE_FILL]
bench = Benchmarking(algorithms,
                     sys_sizes=sizes,
                     target_configs=target_configs,
                     error_models_list = err_models,
                     phys_params_list=phys_params_list,
                     n_shots=n_shots_4,
                     n_species = 1)
bench.run()

In [ ]:
bench.benchmarking_results

In [ ]:
# extracting success rates
X, Y = np.meshgrid(np.array(tweezer_errors)*100, np.array(vac_lifetimes))
rates = bench.benchmarking_results['success rate'].to_numpy()[1,0,0,:,0,0]
rates_arr = bench.benchmarking_results['success rate'].to_numpy()[:,0,0,:,0,0]

#### Plotting data for Fig. 4 (must run one of the 'Loading data from the paper' or 'Rerunning Fig. 4' boxes first).

In [ ]:
fig4 = plt.figure(figsize=(4.75,8.25))

outer_gs = gridspec.GridSpec(nrows=3, ncols=2, figure=fig4, width_ratios=[30, 1], height_ratios=[0.9, 1, 1], hspace=0.4, wspace=0.1)

# First row: nested GridSpec for two plots
inner_gs = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer_gs[0, 0], wspace=0.3)

ax1 = fig4.add_subplot(inner_gs[0, 0])
ax2 = fig4.add_subplot(inner_gs[0, 1])
cax2 = fig4.add_subplot(outer_gs[0, 1])  # Colorbar axis
# Second row: single plot
ax3 = fig4.add_subplot(outer_gs[1, 0])
cax3 = fig4.add_subplot(outer_gs[1, 1])

# Third row: single plot
ax4 = fig4.add_subplot(outer_gs[2, 0])

text_size = 10
markersize = 6
lifetime_linecut_ind = 8

cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", [_nikhilorange, 'white', _nikhilblue])
linecut_col = _nikhilgreen
tickpad = 1

for i in range(3):
    try:
        rates = rates_arr[i,:]
    except IndexError:
        rates = np.zeros_like(rates_arr[0,:]).reshape(len(vac_lifetimes), len(tweezer_errors))
        for mi in range(len(rates)):
            for mj in range(len(rates[0])):
                options = rates_arr[:,mi*len(tweezer_errors)+mj]
                rates[mi,mj] = options[0]/options[1]
    Z = rates.reshape(len(tweezer_errors),len(vac_lifetimes)).T
    if i == 2:
        vcenter = 1
        vmax = Z.max()
        Z_smooth = gaussian_filter(Z, sigma=1)
    else:
        vcenter = 0.5
        vmax = 1

    norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=vcenter, vmax=vmax)
    if i==0:
        im = ax1.pcolor(X,Y, Z, cmap=cmap, norm=norm)
        ax1.set_yscale('log')
        ax1.set_xscale('log')
        ax1.set_ylim([np.min(vac_lifetimes), np.max(vac_lifetimes)])
        ax1.set_ylabel("Lifetime [s]", fontsize=text_size)
        ax1.yaxis.set_label_coords(-0.5, -0.35, transform=None)
        ax1.set_title('Hun.', fontsize=text_size)
        ax1.set_xticks([5e-2, 1e-1, 5e-1])
        ax1.set_xticklabels([r'0.05', r'0.1', r'0.5'],fontsize = text_size)
        ax1.set_yticks([2, 20, 200])
        ax1.set_yticklabels(['2', '20', '200'],fontsize = text_size)
        ax1.tick_params(direction = 'inout', axis='both', which='major', pad=tickpad)
    elif i == 1:
        im = ax2.pcolor(X,Y, Z, cmap=cmap, norm=norm)
        ax2.set_yscale('log')
        ax2.set_xscale('log')
        ax2.set_ylim([np.min(vac_lifetimes), np.max(vac_lifetimes)])
        ax2.set_title('BC.', fontsize=text_size)
        ax2.set_xticks([5e-2, 1e-1, 5e-1])
        ax2.set_xticklabels([r'0.05', r'0.1', r'0.5'],fontsize = text_size)
        ax2.set_yticks([2, 20, 200])
        cbar = fig4.colorbar(im, cax=cax2)
        cbar.ax.tick_params(labelsize=text_size, direction = 'inout', pad=tickpad)
        cbar.ax.set_yticks([1,0.5,0])
        cbar.ax.set_yticklabels([r'1',r'0.5', r'0'])
        cbar.ax.set_ylabel(r'Success rate r', fontsize=text_size, labelpad = 1.5)
        ax2.tick_params(direction = 'inout', axis='both', pad=tickpad)
    elif i == 2:
        
        print(f'Linecut position: lifetime = {vac_lifetimes[lifetime_linecut_ind]:.2e} s')
        im = ax3.pcolor(X,Y, Z, cmap=cmap, norm=norm)
        ax3.set_yscale('log')
        ax3.set_xscale('log')
        ax3.set_ylim([np.min(vac_lifetimes), np.max(vac_lifetimes)])
        # Normalize so that 1 is at the center
        cbar = fig4.colorbar(im, cax=cax3, cmap=cmap, norm=norm)
        ax3.axhline(vac_lifetimes[lifetime_linecut_ind], linestyle = 'dashdot', color = linecut_col, linewidth = 1)
        ax3.set_xticks([5e-2,1e-1, 5e-1])
        ax3.set_xticklabels([r'0.05', r'0.1', r'0.5'],fontsize = text_size)
        ax3.set_yticks([2, 20, 200])
        ax3.set_yticklabels(['2', '20', '200'],fontsize = text_size)
        ax3.tick_params(direction = 'inout', pad=tickpad, axis='both')
        cbar.ax.set_ylabel('rH/rBC', fontsize=text_size, labelpad = 1.25)
        cbar.ax.tick_params(labelsize=text_size, direction = 'inout', axis='both', pad=tickpad)
        cbar.ax.set_yticks([2,1,0.5,0])
        cbar.ax.set_yticklabels([r'2',r'1',r'0.5', r'0'])
        con = ax3.contour(X,Y,Z_smooth, levels = [0.75, 1, 1.25], colors = 'k', linewidths = 1, linestyles = ['-', '--', ':'])
        ax3.clabel(con, fontsize=text_size-2)
        ax3.set_xlabel("Tweezer error [%]", fontsize=text_size)

pointcolor = linecut_col

ax4.errorbar(np.array(tweezer_errors)*100, rates[:,lifetime_linecut_ind], yerr=1/np.sqrt(n_shots_4)*rates[:,lifetime_linecut_ind], c=pointcolor,fmt = "v", markersize = markersize, markerfacecolor=mpl.colors.to_rgba(pointcolor, alpha=0.4), markeredgecolor=mpl.colors.to_rgba(pointcolor, alpha=0.7), elinewidth=1)
ax4.set_xscale('log')
ax4.hlines(1, xmin = min(tweezer_errors)*0.8*100, xmax = max(tweezer_errors)*100, color = 'k', ls = '--', linewidth = 1)
ax4.set_xlabel('Tweezer error [%]', fontsize = text_size)
ax4.set_xticks([0.05, 0.1, 0.5])
ax4.set_xticklabels(['0.05', '0.1', '0.5'],fontsize = text_size)
ax4.set_yticks([0.5, 1, 1.5])
ax4.set_yticklabels(['0.5','1.0', '1.5'],fontsize = text_size)
ax4.set_ylabel('rH / rBC', fontsize = text_size)
ax4.tick_params(labelsize=text_size, pad=tickpad, direction='inout')

plt.subplots_adjust(bottom=0.17, right = 0.78, left = 0.22, top = 0.83)

## Figure 5

#### Loading data from the paper

In [ ]:
# Fig 5c
saved_bench_5c = Benchmarking()
saved_bench_5c.load('fig5c_final_260507')
inout_success = saved_bench_5c.benchmarking_results['success rate'].to_numpy()[0,:,0,0,0,0]
naive_success = saved_bench_5c.benchmarking_results['success rate'].to_numpy()[1,:,0,0,0,0]
raw_n_targets = saved_bench_5c.benchmarking_results['n targets'].to_numpy()[0,:,0,0,0,0]
n_targets = np.array([ast.literal_eval(x) for x in raw_n_targets], dtype=np.int64)

n_atoms = []
for leng in n_targets:
    n_atoms.append(leng[0])

n_shots_5c = 1000

# Fig 5d,e
saved_bench_5de = Benchmarking()
saved_bench_5de.load('fig5de_260513.nc')

loading_prob_range = np.array([0.1, 0.2, 0.225, 0.25, 0.275, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.925, 0.95, 0.975, 1])
n_shots_5de = 400

inout_checker_rate = saved_bench_5de.benchmarking_results['success rate'].to_numpy()[0,0,0,0,:,0]
inout_zebra_rate =   saved_bench_5de.benchmarking_results['success rate'].to_numpy()[0,1,0,0,:,0]
inout_separ_rate =   saved_bench_5de.benchmarking_results['success rate'].to_numpy()[0,2,0,0,:,0]
sufficient_rate =    saved_bench_5de.benchmarking_results['sufficient rate'].to_numpy()[0,0,0,0,:,0]

naive_checker_rate = saved_bench_5de.benchmarking_results['success rate'].to_numpy()[1,0,0,0,:,0]
naive_zebra_rate =   saved_bench_5de.benchmarking_results['success rate'].to_numpy()[1,1,0,0,:,0]
naive_separ_rate =   saved_bench_5de.benchmarking_results['success rate'].to_numpy()[1,2,0,0,:,0]

inout_checker_time = saved_bench_5de.benchmarking_results['time'].to_numpy()[0,0,0,0,:,0]
inout_zebra_time =   saved_bench_5de.benchmarking_results['time'].to_numpy()[0,1,0,0,:,0]
inout_separ_time =   saved_bench_5de.benchmarking_results['time'].to_numpy()[0,2,0,0,:,0]

naive_checker_time = saved_bench_5de.benchmarking_results['time'].to_numpy()[1,0,0,0,:,0]
naive_zebra_time =   saved_bench_5de.benchmarking_results['time'].to_numpy()[1,1,0,0,:,0]
naive_separ_time =   saved_bench_5de.benchmarking_results['time'].to_numpy()[1,2,0,0,:,0]

#### Rerunning Fig. 5c
Runtime: ~ 4.5 hrs for `n_shots_5c = 1000`.

In [ ]:
n_shots_5c = 1000 

# experimental parameters
array = movr.AtomArray(n_species=2, shape=[20,20])
target_storage = []

# Generate target configs with different sizes
for len in range(2,13,2):
    array.generate_target(movr.Configurations.SEPARATE, middle_size=[len,len])
    target_storage.append(array.target)
targets = np.array([target_storage], dtype = 'object')


bench_5c = Benchmarking(algos=[algos.InsideOut(), algos.NaiveParHung()],
                        sys_sizes=[20],
                        n_shots = n_shots_5c, 
                        n_species=2,
                        target_configs=targets)
bench_5c.run()

In [ ]:
bench_5c.benchmarking_results

In [ ]:
# Record the success rate for plotting
inout_success = bench_5c.benchmarking_results['success rate'].to_numpy()[0,:,0,0,0,0]
naive_success = bench_5c.benchmarking_results['success rate'].to_numpy()[1,:,0,0,0,0]
target_len = bench_5c.benchmarking_results['n targets'].to_numpy()[0,:,0,0,0,0]

n_atoms = []
for leng in target_len:
    n_atoms.append(leng[0])

#### Rerunning Fig. 5d, 5e
Runtime: ~12 hrs for `n_shots_5de = 400`.

In [ ]:
n_shots_5de = 400

# experimental parameters
array = movr.AtomArray(n_species=2, shape=[20, 20])
target_storage = []
target_list = [movr.Configurations.CHECKERBOARD, movr.Configurations.ZEBRA_HORIZONTAL, movr.Configurations.SEPARATE]

for config in target_list:
    array.generate_target(config, middle_size=[10,10])
    target_storage.append(array.target)

targets = np.array([target_storage], dtype = 'object')

loading_prob_range = np.array([0.1, 0.2, 0.225, 0.25, 0.275, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.925, 0.95, 0.975, 1])
loading_sweep_storage = []
for load_prob in loading_prob_range:
    loading_params = movr.PhysicalParams(loading_prob= load_prob, AOD_speed = 0.2)
    loading_sweep_storage.append(loading_params)

error_models = [movr.ZeroNoise(200e-6, 200e-6, 12.5e-6, 12.5e-6)]
bench_5de = Benchmarking(algos=[algos.InsideOut(), algos.NaiveParHung()],
                         sys_sizes=[20], 
                         n_shots = n_shots_5de,
                         n_species=2, 
                         phys_params_list = loading_sweep_storage,
                         error_models_list = error_models,
                         target_configs = targets,
                         check_sufficient_atoms = False)

bench_5de.run()

In [ ]:
bench_5de.benchmarking_results

In [ ]:
inout_checker_rate = bench_5de.benchmarking_results['success rate'].to_numpy()[0,0,0,0,:,0]
inout_zebra_rate = bench_5de.benchmarking_results['success rate'].to_numpy()[0,1,0,0,:,0]
inout_separ_rate = bench_5de.benchmarking_results['success rate'].to_numpy()[0,2,0,0,:,0]
sufficient_rate = bench_5de.benchmarking_results['sufficient rate'].to_numpy()[0,0,0,0,:,0]

naive_checker_rate = bench_5de.benchmarking_results['success rate'].to_numpy()[1,0,0,0,:,0]
naive_zebra_rate = bench_5de.benchmarking_results['success rate'].to_numpy()[1,1,0,0,:,0]
naive_separ_rate = bench_5de.benchmarking_results['success rate'].to_numpy()[1,2,0,0,:,0]

inout_checker_time = bench_5de.benchmarking_results['time'].to_numpy()[0,0,0,0,:,0]
inout_zebra_time = bench_5de.benchmarking_results['time'].to_numpy()[0,1,0,0,:,0]
inout_separ_time = bench_5de.benchmarking_results['time'].to_numpy()[0,2,0,0,:,0]

naive_checker_time = bench_5de.benchmarking_results['time'].to_numpy()[1,0,0,0,:,0]
naive_zebra_time = bench_5de.benchmarking_results['time'].to_numpy()[1,1,0,0,:,0]
naive_separ_time = bench_5de.benchmarking_results['time'].to_numpy()[1,2,0,0,:,0]

#### Plotting data for Fig. 5 (must run one of the 'Loading data from the paper' or 'Rerunning Fig. 5' boxes first).

In [ ]:
alpha_marker = 0.4
alpha_border = 0.7
colors = [_nikhilgreen, _quantumgray, _nikhilblue, _quantumviolet, _nikhilorange, [0.4, 0.2, 0.7]] #['orange', '#555555', 'b', '#008080']
facecolors = colors

shapemarkers = ["v", "s", "o", "d", '^', "P"]
markers = shapemarkers
mark_size = 6
labels_c = ['InOut', 'ParHun']
labels_d = ['Sufficient rate','Checker (InOut)', 'Zebra (InOut)', 'Zones (InOut)', 'Checker (ParHun)', 'Zebra (ParHun)', 'Zones (ParHun)']

edgecolors = colors

fig5 = plt.figure(figsize = (10,3))

# Outer GridSpec: 1 row × 4 columns
outer_gs = gridspec.GridSpec(1, 3, width_ratios=[0.7, 1, 1], wspace=0.4)

# First column: Nested GridSpec (2 rows in col 0)
left_gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer_gs[0], height_ratios=[1, 1.5], hspace=0.1)

# Bottom half of first column: single plot
axC = fig5.add_subplot(left_gs[1])

# Columns 2–4: individual plots
axD = fig5.add_subplot(outer_gs[1])
axE = fig5.add_subplot(outer_gs[2])

# # fig 5c
axC.errorbar(n_atoms, inout_success, yerr=inout_success/np.sqrt(n_shots_5c),c=colors[0],label = labels_c[0], fmt = markers[0], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[0], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[0], alpha=alpha_border))
axC.errorbar(n_atoms, naive_success, yerr=naive_success/np.sqrt(n_shots_5c),c=colors[1],label = labels_c[1], fmt = markers[1], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[1], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[1], alpha=alpha_border))
text_size = 10
axC.legend(prop={'size': text_size})
axC.set_ylim(0,1.05)
axC.set_yticks([0,0.25,0.5,0.75,1])
axC.set_ylabel('Success rate', fontsize=text_size)
axC.set_xlabel('Number of atoms', fontsize=text_size)
axC.grid(True, which="major", linewidth=0.8)

# fig 5d
load_probs = loading_prob_range*100
# sufficient atom rate
axD.plot(load_probs, sufficient_rate, label = labels_d[0], ls='--', c = 'k',linewidth=0.5)
# inside out
axD.errorbar(load_probs, y=inout_checker_rate, yerr=inout_checker_rate/np.sqrt(n_shots_5de), c=colors[0],label = labels_d[1], fmt = markers[0], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[0], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[0], alpha=alpha_border))
axD.errorbar(load_probs, y=inout_zebra_rate,   yerr=inout_zebra_rate/np.sqrt(n_shots_5de), c=colors[1],label = labels_d[2], fmt = markers[0], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[1], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[1], alpha=alpha_border))
axD.errorbar(load_probs, y=inout_separ_rate,   yerr=inout_separ_rate/np.sqrt(n_shots_5de), c=colors[2],label = labels_d[3], fmt = markers[0], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[2], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[2], alpha=alpha_border))
# par hun
axD.errorbar(load_probs, y=naive_checker_rate, yerr=naive_checker_rate/np.sqrt(n_shots_5de), c=colors[0],label = labels_d[4], fmt = markers[1], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[0], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[0], alpha=alpha_border))
axD.errorbar(load_probs, y=naive_zebra_rate,   yerr=naive_zebra_rate/np.sqrt(n_shots_5de), c=colors[1],label = labels_d[5], fmt = markers[1], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[1], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[1], alpha=alpha_border))
axD.errorbar(load_probs, y=naive_separ_rate,   yerr=naive_separ_rate/np.sqrt(n_shots_5de), c=colors[2],label = labels_d[6], fmt = markers[1], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[2], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[2], alpha=alpha_border))
axD.set_ylim(0,1.05)
axD.set_yticks([0,0.25,0.5,0.75,1])
axD.set_xticks([0,25,50,75,100])
axD.set_ylabel('Success rate', fontsize=text_size)
axD.set_xlabel('Loading probability [%]', fontsize=text_size)
axD.grid(True, which="major", linewidth=0.8)
# axD.legend(prop={'size': text_size})

# fig 5e
load_probs = loading_prob_range*100
start_ind = 4
end_ind_parh_separ = -4
# inside out
axE.errorbar(load_probs[start_ind:], y=inout_checker_time[start_ind:]*1e3, yerr=inout_checker_time[start_ind:]*1e3/np.sqrt(inout_checker_rate[start_ind:]*n_shots_5de), c=colors[0],label = labels_d[1], fmt = markers[0], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[0], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[0], alpha=alpha_border))
axE.errorbar(load_probs[start_ind:], y=inout_zebra_time[start_ind:]*1e3, yerr=inout_zebra_time[start_ind:]*1e3/np.sqrt(inout_zebra_rate[start_ind:]*n_shots_5de), c=colors[1],label = labels_d[2], fmt = markers[0], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[1], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[1], alpha=alpha_border))
axE.errorbar(load_probs[start_ind:], y=inout_separ_time[start_ind:]*1e3,yerr=inout_separ_time[start_ind:]*1e3/np.sqrt(inout_separ_rate[start_ind:]*n_shots_5de), c=colors[2],label = labels_d[3], fmt = markers[0], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[2], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[2], alpha=alpha_border))
# par hun
axE.errorbar(load_probs[start_ind:], y=naive_checker_time[start_ind:]*1e3, yerr=naive_checker_time[start_ind:]*1e3/np.sqrt(naive_checker_rate[start_ind:]*n_shots_5de), c=colors[0],label = labels_d[4], fmt = markers[1], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[0], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[0], alpha=alpha_border))
axE.errorbar(load_probs[start_ind:], y=naive_zebra_time[start_ind:]*1e3,   yerr=naive_zebra_time[start_ind:]*1e3/np.sqrt(naive_zebra_rate[start_ind:]*n_shots_5de), c=colors[1],label = labels_d[5], fmt = markers[1], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[1], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[1], alpha=alpha_border))
axE.errorbar(load_probs[start_ind:end_ind_parh_separ], y=naive_separ_time[start_ind:end_ind_parh_separ]*1e3,   yerr=naive_separ_time[start_ind:end_ind_parh_separ]*1e3/np.sqrt(naive_separ_rate[start_ind:end_ind_parh_separ]*n_shots_5de), c=colors[2],label = labels_d[6], fmt = markers[1], markersize = mark_size, markerfacecolor=mpl.colors.to_rgba(facecolors[2], alpha=alpha_marker), markeredgecolor=mpl.colors.to_rgba(edgecolors[2], alpha=alpha_border))

axE.set_ylabel('Rearrangement time [ms]', fontsize=text_size)
axE.set_xlabel('Loading probability [%]', fontsize=text_size)
axE.set_yscale('log')
axE.set_xticks([25,50,75,100])
axE.set_yticks([30, 50,100,200], labels = ['30','50','100','200'])
axE.yaxis.set_minor_formatter(NullFormatter())
# optional: remove minor tick marks too
# axE.yaxis.set_minor_locator(NullLocator())
axE.grid(True, which="major", linewidth=0.8)

#axE.legend(prop={'size': text_size})